In [1]:

from pathlib import Path
import time

import numpy as np
import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import Binarizer, OneHotEncoder, PCA, SQLTransformer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# keep the pandas tables wide enough to read in the notebook.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# store the project paths, the watched stream folder, the stream checkpoint, and the shared seed in one place.
PROJECT_DIR = Path('/Users/alexdevoid/Documents/Stats/ST554-HW/FinalProject')

# start Spark session
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('final_project_power_stream')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
spark.conf.set('spark.sql.shuffle.partitions', '8')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/24 23:11:50 WARN Utils: Your hostname, Alexs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.101 instead (on interface en0)
26/04/24 23:11:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 23:11:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:

# read data from URL into pandas
power_pdf = pd.read_csv('https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv')

# convert into a cached Spark SQL DataFrame.
power_sdf = spark.createDataFrame(power_pdf).cache()
_ = power_sdf.count()

# summarize the row count, column count, and response column
data_summary = pd.DataFrame(
    {
        'rows': [power_pdf.shape[0]],
        'columns': [power_pdf.shape[1]],
        'response': ['Power_Zone_3'],
    }
)


# show summary
display(data_summary)
# how first rows of the Spark SQL DataFrame
display(power_pdf.head())

,rows,columns,response
0,47174,10,Power_Zone_3


,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [3]:
dtype_summary = pd.DataFrame(power_sdf.dtypes, columns=['column', 'spark_type'])
dtype_summary

,column,spark_type
0,Temperature,double
1,Humidity,double
2,Wind_Speed,double
3,General_Diffuse_Flows,double
4,Diffuse_Flows,double
5,Power_Zone_1,double
6,Power_Zone_2,double
7,Power_Zone_3,double
8,Month,bigint
9,Hour,bigint


In [4]:
# cast Hour to double and rename Power_Zone_3 to label
sql_transform = SQLTransformer(
    statement="""
    SELECT *,
           CAST(Hour AS DOUBLE) AS Hour_double,
           Power_Zone_3 AS label
    FROM __THIS__
    """
)

# binarize the cast Hour column using a 6.5 threshold
hour_binarizer = Binarizer(inputCol='Hour_double', outputCol='hour_binary', threshold=6.5)

# weather variables in one vector
weather_assembler = VectorAssembler(
    inputCols=['Temperature', 'Humidity', 'Wind_Speed', 'General_Diffuse_Flows', 'Diffuse_Flows'],
    outputCol='weather_features'
)

# reduce to two principal-component scores.
weather_pca = PCA(k=2, inputCol='weather_features', outputCol='weather_pcs')

# one-hot encode Month
month_encoder = OneHotEncoder(inputCols=['Month'], outputCols=['Month_vec'])

# combine
feature_assembler = VectorAssembler(
    inputCols=['weather_pcs', 'hour_binary', 'Power_Zone_1', 'Power_Zone_2', 'Month_vec'],
    outputCol='features'
)

# elastic-net linear regression model
elastic_net = LinearRegression(featuresCol='features', labelCol='label', predictionCol='prediction')

# wrap the transformations and model into a pipeline.
power_pipeline = Pipeline(
    stages=[
        sql_transform,
        hour_binarizer,
        weather_assembler,
        weather_pca,
        month_encoder,
        feature_assembler,
        elastic_net,
    ]
)

# build the grid
param_values = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]
param_grid = (
    ParamGridBuilder()
    .addGrid(elastic_net.regParam, param_values)
    .addGrid(elastic_net.elasticNetParam, param_values)
    .build()
)

# use RMSE as the CV metric for the elastic-net fit.
rmse_evaluator = RegressionEvaluator(labelCol='label', predictionCol='prediction', metricName='rmse')


# 5-fold cross-validation over the pipeline.
RANDOM_STATE = 554
power_cv = CrossValidator(
    estimator=power_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=rmse_evaluator,
    numFolds=5,
    parallelism=1,
    seed=RANDOM_STATE,
)

# fit the full cross-validated pipeline on the power data.
power_cv_model = power_cv.fit(power_sdf)

# find which tuning variable combination gave the smallest mean CV RMSE.
best_idx = int(np.argmin(power_cv_model.avgMetrics))
best_params = {param.name: value for param, value in param_grid[best_idx].items()}
cv_rmse = float(power_cv_model.avgMetrics[best_idx])

# score the data with the fitted model
training_predictions = power_cv_model.transform(power_sdf)
training_rmse = float(rmse_evaluator.evaluate(training_predictions))

# collect the tuning and error results into a table.
model_summary = pd.DataFrame(
    {
        'best_regParam': [float(best_params['regParam'])],
        'best_elasticNetParam': [float(best_params['elasticNetParam'])],
        'cv_rmse': [cv_rmse],
        'training_rmse': [training_rmse],
    }
)

display(model_summary)

,best_regParam,best_elasticNetParam,cv_rmse,training_rmse
0,0.05,0.5,2147.84652,2147.097346
